In [1]:
import sys

sys.path.append("../src/")

from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
from hydra.utils import instantiate
from pathlib import Path
import os
from matplotlib.animation import FuncAnimation

from utils.train_utils import extract_wet
from utils.data_utils import (
    get_train_test_ranges,
    data_CNN_Disk,
    data_CNN_Disk_steps,
    gen_3D_data,
)
from utils.eval_utils import (
    generate_model_rollout,
    compute_KE,
    compute_nino34,
    compute_amo,
    gen_KE_range,
    gen_value_range,
    compute_mean_single,
)
from utils.climate_utils import compute_laplacian_wet

import numpy as np
import torch
import xarray as xr
import copy

from hydra import compose, initialize_config_dir
import copy
from datetime import datetime
import os

In [2]:
# ########################################################
# 1 
# All Levels - Hist = 1, HFDS Anom, 100 year emulation, 1975, NetZeroHf
# with initialize_config_dir(
#     version_base=None,
#     config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
# ):
#     args = compose(
#         config_name="exp/eval_unet_global_3D_all_hfds_anoms",
#         overrides=[
#             "output_dir=./temp/{0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHf1975Epochs70Epoch55Years100_10repeat_test".format(
#                 str(datetime.now())[:10]
#             ),
#             "network={0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHf1975Epochs70Epoch55Years100_10repeat_test".format(
#                 str(datetime.now())[:10]
#             ),
#             "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train_3D/2024-09-09-convnextunet_v021_hist1_hfds_anom_1975/hfds/saved_nets/convnextunet_epoch_55_steps_4_global_3D_all_N_train_2850_Lateral_Data_025_no_smooth.pt]",
#             "hist=1",
#             "unet.ch_width=[157,200,250,300,400]",
#             "run_gen_pred=True",
#             "N_samples=0",
#             "N_val=0",
#             "N_test=7200",
#             "data_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_netzerohfds",
#             "data_means_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_means",
#             "data_stds_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_stds",
#             "pred_names=null",
#             "pred_paths=null",
            # "+dataset_name=OM4_1990_repeat",
            # "+save_factor=40",
            # "train_region=global_3D",
#             "region=global_3D",
#             "model_name_replace=Convnext",
#             "depth_mode=all",
#         ],
#     )

# ########################################################
# 2
# All Levels - Hist = 1, HFDS Anom, 100 year emulation, 1975, NetZeroHf, Climate Change Forcing
# with initialize_config_dir(
#     version_base=None,
#     config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
# ):
#     args = compose(
#         config_name="exp/eval_unet_global_3D_all_hfds_anoms",
#         overrides=[
#             "output_dir=./temp/{0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x".format(
#                 str(datetime.now())[:10]
#             ),
#             "network={0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x".format(
#                 str(datetime.now())[:10]
#             ),
#             "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train_3D/2024-09-09-convnextunet_v021_hist1_hfds_anom_1975/hfds/saved_nets/convnextunet_epoch_55_steps_4_global_3D_all_N_train_2850_Lateral_Data_025_no_smooth.pt]",
#             "hist=1",
#             "unet.ch_width=[157,200,250,300,400]",
#             "run_gen_pred=True",
#             "N_samples=0",
#             "N_val=0",
#             "N_test=7200",
#             "data_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_netzerohfds_nojump3xcc",
#             "data_means_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_means",
#             "data_stds_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_stds",
#             "pred_names=null",
#             "pred_paths=null",
#             "+dataset_name=OM4_1990_repeat",
#             "+save_factor=40",
#             "train_region=global_3D",
#             "region=global_3D",
#             "model_name_replace=Convnext",
#             "depth_mode=all",
#         ],
#     )

# ########################################################
# 3
# All Levels - Hist = 1, HFDS Anom, 100 year emulation, 1975, TempOnly, NetZeroHf
# with initialize_config_dir(
#     version_base=None,
#     config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
# ):
#     args = compose(
#         config_name="exp/eval_unet_global_3D_all_hfds_anoms",
#         overrides=[
#             "output_dir=./temp/{0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnly1975Epochs70Epoch55Years100_10repeat_36_6k".format(
#                 str(datetime.now())[:10]
#             ),
#             "network={0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnly1975Epochs70Epoch55Years100_10repeat_36_6k".format(
#                 str(datetime.now())[:10]
#             ),
#             "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train_3D/2024-09-11-convnextunet_v021_hist1_hfds_anom_1975_nofast/convnextunet_epoch_55_beststeps_4_global_3D_all_N_train_2850_Lateral_Data_025_no_smooth.pt]",
#             "hist=1",
#             "unet.ch_width=[157,200,250,300,400]",
#             "run_gen_pred=True",
#             "N_samples=0",
#             "N_val=0",
#             "N_test=29200",
#             "data_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_netzerohfds",
#             "data_means_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_means",
#             "data_stds_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_stds",
#             "exp_num_in=3D_noFast_all",
#             "exp_num_out=3D_noFast_all",
#             "pred_names=null",
#             "pred_paths=null",
            # "+dataset_name=OM4_1990_repeat",
            # "+save_factor=200",
            # "train_region=global_3D",
#             "region=global_3D",
#             "model_name_replace=Convnext",
#             "depth_mode=all",
#         ],
#     )

# ########################################################
# 4
# All Levels - Hist = 1, HFDS Anom, 100 year emulation, 1975, TempOnly, NetZeroHf, Climate Change Forcing
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global_3D_all_hfds_anoms",
        overrides=[
            "output_dir=./temp/{0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnlyNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x".format(
                str(datetime.now())[:10]
            ),
            "network={0}_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnlyNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x".format(
                str(datetime.now())[:10]
            ),
            "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train_3D/2024-09-11-convnextunet_v021_hist1_hfds_anom_1975_nofast/convnextunet_epoch_55_beststeps_4_global_3D_all_N_train_2850_Lateral_Data_025_no_smooth.pt]",
            "hist=1",
            "unet.ch_width=[157,200,250,300,400]",
            "run_gen_pred=True",
            "N_samples=0",
            "N_val=0",
            "N_test=29200",
            "data_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_netzerohfds_nojump3xcc",
            "data_means_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_means",
            "data_stds_zarr=3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_stds",
            "exp_num_in=3D_noFast_all",
            "exp_num_out=3D_noFast_all",
            "pred_names=null",
            "pred_paths=null",
            "+dataset_name=OM4_1990_repeat",
            "+save_factor=200", # 20
            "train_region=global_3D",
            "region=global_3D",
            "model_name_replace=Convnext",
            "depth_mode=all",
        ],
    )

In [3]:
inputs_str = INPT_VARS[args.exp_num_in]
extra_in_str = EXTRA_VARS[args.exp_num_extra]
outputs_str = OUT_VARS[args.exp_num_out]
levels = args.exp_num_in.split("_")[-1]
if "all" in levels:
    levels = 19
elif "2D" in levels:
    levels = 1
else:
    levels = int(levels)

str_in = "".join([i + "_" for i in inputs_str])
str_ext = "".join([i + "_" for i in extra_in_str])
str_out = "".join([i + "_" for i in outputs_str])

print("inputs: " + str_in)
print("extra inputs: " + str_ext)
print("outputs: " + str_out)
print("levels: " + str(levels))

N_atm = len(extra_in_str)  # Number of atmosphere variables
N_in = len(inputs_str)
if args.lateral:
    N_extra = (
        N_atm + N_in
    )  # Number of atmosphere variables + Lateral boundary variables
else:
    N_extra = N_atm  # Number of atmosphere variables
N_out = len(outputs_str)

num_in = int((args.hist + 1) * N_in + N_extra)
num_out = int((args.hist + 1) * len(outputs_str))

print("Number of inputs: ", num_in)  # 3 (ocean speeds + ocean temp)(t) +
# 3 (atm wind stresses + atm temp)(t) +
# 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
print("Number of outputs: ", num_out)  # 3

if "swin" in args.network.lower():
    pass
    # model = instantiate(
    #     args.swin,
    #     in_channels=num_in,
    #     output_channels=num_out,
    #     pretrain_img_size=[180, 360],
    #     wet=wet.cuda(),
    #     hist=args.hist,
    # )
elif "convnext" in args.network.lower():
    if args.unet.ch_width[0] != num_in:
        print(
            "Changing ch_width to match number of inputs {0} -> {1}".format(
                args.unet.ch_width[0], num_in
            )
        )
        args.unet.ch_width[0] = num_in

# Post-fix strings
str_train = (
    "steps_"
    + str(args.steps)
    + "_"
    + args.train_region
    + "_"
    + args.depth_mode
    + "_N_train_4000"
    + "_Lateral_Data_025_no_smooth"
)
str_save = (
    "steps_"
    + str(args.steps)
    + "_"
    + args.train_region
    + "_"
    + args.region
    + "_"
    + args.depth_mode
    + "+N_samples_"
    + str(args.N_samples)
)
post_model_name = (
    "Train_"
    + args.train_region
    + "_Test_"
    + args.region
    + "_"
    + args.depth_mode
    + "_N_train_"
    + str(args.N_samples)
    + "_Lateral_Data_025_no_smooth"
)
post_pred_name = (
    args.region + "_" + args.depth_mode + "_N_samples_" + str(args.N_samples)
)

# Getting start and end indices of train and test
s_train, e_train, e_test = get_train_test_ranges(
    args.N_samples, args.N_val, args.lag, args.hist, args.interval
)
dataset_name = args.dataset_name

if "OM4" in dataset_name:
    timestep_str = "\\times 5"
else:
    raise ValueError("Dataset not recognized")


print("Calculating mask tensors")
wet_zarr = xr.open_zarr(os.path.join("/vast/sd5313/data/m2lines/3D_ocean_data/", args.wet_file))
wet = extract_wet(wet_zarr, outputs_str, args.hist)
print("Wet resolution:", wet.shape)
print("e_test: ", e_test)

inputs: thetao_lev_2_5_thetao_lev_10_0_thetao_lev_22_5_thetao_lev_40_0_thetao_lev_65_0_thetao_lev_105_0_thetao_lev_165_0_thetao_lev_250_0_thetao_lev_375_0_thetao_lev_550_0_thetao_lev_775_0_thetao_lev_1050_0_thetao_lev_1400_0_thetao_lev_1850_0_thetao_lev_2400_0_thetao_lev_3100_0_thetao_lev_4000_0_thetao_lev_5000_0_thetao_lev_6000_0_so_lev_2_5_so_lev_10_0_so_lev_22_5_so_lev_40_0_so_lev_65_0_so_lev_105_0_so_lev_165_0_so_lev_250_0_so_lev_375_0_so_lev_550_0_so_lev_775_0_so_lev_1050_0_so_lev_1400_0_so_lev_1850_0_so_lev_2400_0_so_lev_3100_0_so_lev_4000_0_so_lev_5000_0_so_lev_6000_0_zos_
extra inputs: tauuo_tauvo_hfds_hfds_anomalies_
outputs: thetao_lev_2_5_thetao_lev_10_0_thetao_lev_22_5_thetao_lev_40_0_thetao_lev_65_0_thetao_lev_105_0_thetao_lev_165_0_thetao_lev_250_0_thetao_lev_375_0_thetao_lev_550_0_thetao_lev_775_0_thetao_lev_1050_0_thetao_lev_1400_0_thetao_lev_1850_0_thetao_lev_2400_0_thetao_lev_3100_0_thetao_lev_4000_0_thetao_lev_5000_0_thetao_lev_6000_0_so_lev_2_5_so_lev_10_0_so_lev_22

In [4]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from scipy.ndimage import gaussian_filter
from einops import rearrange
import os

class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        ind_start,
        long_rollout,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.ind_start = ind_start

        assert self.interval == 1
        assert self.lag == 1

        data = data.isel(time=slice(self.ind_start, None))
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]

        # This class will be used only for validation and rollouts
        # Rolling indices to keep track of histories/past states:
        # HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
        # HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[2, 3], [4, 5]]; 2->[[4, 5], [6, 7]]; 3->[[6, 7], [8, 9]]
        # HIST=2 ; 0->[[0, 1, 2], [3, 4, 5]]; 1->[[3, 4, 5], [6, 7, 8]]; 2->[[6, 7, 8], [9, 10, 11]]; 3->[[9, 10, 11], [12, 13, 14]]
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        total_steps = 2 * self.hist + 1
        rolling_indices = (
            indices.rolling(time=len(data.time) - total_steps, center=False)
            .construct("window_dim")
            .astype(int)
        )
        rolling_indices = rolling_indices.transpose("window_dim", "time").isel(
            time=slice(len(data.time) - total_steps - 1, None)
        )  # Remove first few null indices
        self.rolling_indices = rolling_indices.isel(
            window_dim=slice(0, None, self.hist + 1)
        )  # Skip indices based on history

        if long_rollout:
            window0 = self.rolling_indices.isel(window_dim=0)
            print(
                "Long rollout will begin with input and produce output from time index {0} and {1} respectively".format(
                    window0.isel(time=0).values + ind_start,
                    window0.isel(time=self.hist + 1).values + ind_start,
                )
            )

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]
        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)
        elif type(idx) == int:
            idx = slice(idx, idx + 1, 1)

        rolling_idx = self.rolling_indices.isel(window_dim=idx)
        x_index = xr.Variable(
            ["window_dim", "time"], rolling_idx
        )
        print("Out: ", (self.ind_start + x_index.isel(time=slice(self.hist + 1, None))).values, end=' ')
        data_in = self.inputs_no_extra.isel(time=x_index).isel(
            time=slice(None, self.hist + 1)
        )
        data_in = (
            (data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std
        ).fillna(0)
        data_in = (
            data_in.to_array()
            .transpose("window_dim", "time", "variable", "y", "x")
            .to_numpy()
        )
        data_in = rearrange(
            data_in, "window_dim time variable y x -> window_dim (time variable) y x"
        )
        if len(self.extras.variables) != 0:
            data_in_boundary = self.extras.isel(time=x_index).isel(time=self.hist)
            data_in_boundary = (
                (data_in_boundary - self.extras_mean) / self.extras_std
            ).fillna(0)
            data_in_boundary = (
                data_in_boundary.to_array()
                .transpose("window_dim", "variable", "y", "x")
                .to_numpy()
            )
            data_in = np.concatenate((data_in, data_in_boundary), axis=1)

        label = self.outputs.isel(time=x_index).isel(time=slice(self.hist + 1, None))
        label = ((label - self.out_mean) / self.out_std).fillna(0)
        label = (
            label.to_array()
            .transpose("window_dim", "time", "variable", "y", "x")
            .to_numpy()
        )
        label = rearrange(
            label, "window_dim time variable y x -> window_dim (time variable) y x"
        )

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

In [5]:
# 0.0136986301 / 4

In [6]:
import xarray as xr

assert args.depth_mode == "surface" or args.depth_mode == "all"

data = xr.open_zarr(os.path.join("/vast/as15415/Data", args.data_zarr))
if args.data_zarr== "3D_data_OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_netzerohfds_nojump3xcc":
    print("Updating climate forced runs!")
    data['hfds'] = data['hfds'] + .017462726 
    data['hfds_anomalies'] = data['hfds_anomalies'] + .017462726 
    data['hfds'] = data['hfds'] + np.reshape(np.arange(data.time.size)* (0.0136986301-4.01369026e-04),(-1,1,1))
    data['hfds_anomalies'] = data['hfds_anomalies'] + np.reshape(np.arange(data.time.size)* (0.0136986301-4.01369026e-04),(-1,1,1))


Updating climate forced runs!


In [7]:
repeats = 4
data = xr.concat([data] * repeats, dim="time")
data['time'] = np.arange(data.time.size)
data

<xarray.Dataset>
Dimensions:            (y: 180, x: 360, lev: 19, time: 29200, y_b: 181, x_b: 361)
Coordinates:
    areacello          (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                 (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction     (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time               (time) int64 0 1 2 3 4 ... 29195 29196 29197 29198 29199
    wetmask            (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables: (12/81)
    hfds               (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    hfds_anomalies     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1050_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_105_0       (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_10_0        (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1400_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                 ...
    vo_lev_5000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_550_0       (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_65_0        (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_775_0       (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [8]:

data_mean = xr.open_zarr(
    os.path.join("/vast/sd5313/data/m2lines/3D_ocean_data/", args.data_means_zarr)
)
data_std = xr.open_zarr(
    os.path.join("/vast/sd5313/data/m2lines/3D_ocean_data/", args.data_stds_zarr)
)
train_data = data_CNN_Disk_steps(
    data,
    inputs_str,
    extra_in_str,
    outputs_str,
    wet,
    data_mean,
    data_std,
    args.N_samples,
    args.lag,
    args.interval,
    args.hist,
    args.steps,
    device="cuda",
)

test_data = data_CNN_Disk(
    data,
    inputs_str,
    extra_in_str,
    outputs_str,
    wet,
    data_mean,
    data_std,
    data.time.size,
    args.lag,
    args.interval,
    args.hist,
    e_test,
    long_rollout=True,
    device="cuda",
)
# test_data[0]



/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 1 and 3 respectively


In [9]:
# Model
print("Loading model " + args.network)
if "swin" in args.network.lower():
    model = instantiate(
        args.swin,
        in_channels=num_in,
        output_channels=num_out,
        pretrain_img_size=[180, 360],
        wet=wet.cuda(),
        hist=args.hist,
    )
elif "unet" in args.network.lower():
    model = instantiate(args.unet, n_out=num_out, wet=wet.cuda(), hist=args.hist)

full_model_path = args.ckpt_path
full_model_name = args.network + "_" + post_model_name
output_channels = model.output_channels

model = model.to(args.device)
ckpt_path = args.ckpt_path
model = model

# Stats
mean_in = test_data.in_mean.to_array().to_numpy().reshape(-1)
std_in = test_data.in_std.to_array().to_numpy().reshape(-1)
mean_out = test_data.out_mean.to_array().to_numpy().reshape(-1)
std_out = test_data.out_std.to_array().to_numpy().reshape(-1)

test_data.norm_vals = {
    "s_out": std_out,
    "s_in": std_in,
    "m_out": mean_out,
    "m_in": mean_in,
}

# Getting area tensor
print("Computing area tensor")
grids = xr.open_dataset(os.path.join("/vast/sd5313/data/m2lines/3D_ocean_data/", args.grid_file)).rename({"xu_ocean": "x", "yu_ocean": "y"})

area = torch.from_numpy(grids["area_C"].to_numpy()).to(device="cpu")
pred_model_path = Path("/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds") / full_model_name
if not os.path.isdir(pred_model_path):
    os.makedirs(pred_model_path)

Nb = args.Nb
hist = args.hist
lag = args.lag
N_test = args.N_test
N_samples = args.N_samples
output_dir = args.output_dir
region = args.region
steps = args.steps
network = args.model_name_replace

pred_region = args.region
pred_names = args.pred_names if args.pred_names else []
pred_paths = args.pred_paths if args.pred_paths else []

JUPYTER_MODE = False
    
    


Loading model 2024-09-12_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnlyNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x
Computing area tensor


In [10]:
out_mean = torch.tensor(test_data.out_mean.to_array().to_numpy()).to(device="cuda")
out_std = torch.tensor(test_data.out_std.to_array().to_numpy()).to(device="cuda")

In [11]:
### Generate
def generate_pred_lateral():
    print("Generation Pred begin...")
    for rand_ind, model_path in enumerate(args.ckpt_path):
        print("Random seed: ", rand_ind + 1)
        # try:
        model.load_state_dict(
            torch.load(model_path, map_location=torch.device("cuda"))["model"]
        )
        # except:
        #     model.load_state_dict(
        #         torch.load(model_path, map_location=torch.device("cuda"))
        #     )
        pred_path = pred_model_path / (
                        "Pred_lateral_Fast_Data_025_"
                        + post_pred_name
                        + "_rand_seed_"
                        + str(rand_ind + 1)
                        + ".zarr"
                    )
        save_factor = args.save_factor
        print(f"Using save_factor {save_factor} to produce {N_test // save_factor} steps each iteration")
        outs = None
        if not os.path.isdir(pred_path):
            start = 0
            print(f"Starting save from {start} for Pred path {pred_path}")
            initial_input = None
        else:
            pred = xr.open_zarr(pred_path)
            start = pred.time.size
            print(f"Restarting save from {start} for Pred path {pred_path}")
            last_pred_numpy = pred.isel(time=slice(-2,None)).to_array().to_numpy().squeeze()
            last_pred = torch.tensor(last_pred_numpy).to(device="cuda")
            assert last_pred.shape[:3] == torch.Size([2, 180, 360])
            last_pred = (last_pred - out_mean) / out_std
            initial_input = torch.swapaxes(torch.swapaxes(last_pred, 3, 2), 2, 1).reshape(-1, 180, 360)
            

        for i in range(start, N_test, N_test // save_factor):
            start_time = time.time()
            test_data = data_CNN_Disk(
                data,
                inputs_str,
                extra_in_str,
                outputs_str,
                wet,
                data_mean,
                data_std,
                args.N_test,
                args.lag,
                args.interval,
                args.hist,
                e_test + i,
                long_rollout=True,
                device="cuda",
            )
            
            test_data.norm_vals = {
                "s_out": std_out,
                "s_in": std_in,
                "m_out": mean_out,
                "m_in": mean_in,
            }
            if outs is not None:
                initial_input = outs[-1]
                
            model_pred, outs = generate_model_rollout(
                N_test // save_factor,
                test_data,
                model,
                hist,
                N_in,
                N_extra,
                initial_input,
                Nb,
                region,
            )
            da = xr.DataArray(
                data=model_pred,
                dims=["time", "x", "y", "var"],
            )

            if i == 0:
                da.to_zarr(pred_path, mode="w")
            else:
                da.to_zarr(pred_path, mode="a", append_dim="time")
            print("Saved: ", i, " to ", i + N_test // save_factor)
            print(f"Time taken for generating {N_test // save_factor} predictions: {time.time() - start_time} seconds")


In [ ]:
import time 
start_time = time.time()
if args.run_gen_pred:
    generate_pred_lateral()
    
print(f"Time taken for generating {N_test} predictions: {time.time() - start_time} seconds")

Generation Pred begin...
Random seed:  1
Using save_factor 200 to produce 146 steps each iteration
Restarting save from 7638 for Pred path /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/2024-09-12_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnlyNoJump3xcc1975Epochs70Epoch55Years100_10repeat_newforce30x_Train_global_3D_Test_global_3D_all_N_train_0_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_0_rand_seed_1.zarr


/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 7639 and 7641 respectively
Out:  [[7641 7642]] Out:  [[7641 7642]] Out:  [[7643 7644]] Out:  [[7645 7646]] Out:  [[7647 7648]] Out:  [[7649 7650]] Out:  [[7651 7652]] Out:  [[7653 7654]] Out:  [[7655 7656]] Out:  [[7657 7658]] Out:  [[7659 7660]] Out:  [[7661 7662]] Out:  [[7663 7664]] Out:  [[7665 7666]] Out:  [[7667 7668]] Out:  [[7669 7670]] Out:  [[7671 7672]] Out:  [[7673 7674]] Out:  [[7675 7676]] Out:  [[7677 7678]] Out:  [[7679 7680]] Out:  [[7681 7682]] Out:  [[7683 7684]] Out:  [[7685 7686]] Out:  [[7687 7688]] Out:  [[7689 7690]] Out:  [[7691 7692]] Out:  [[7693 7694]] Out:  [[7695 7696]] Out:  [[7697 7698]] Out:  [[7699 7700]] Out:  [[7701 7702]] Out:  [[7703 7704]] Out:  [[7705 7706]] Out:  [[7707 7708]] Out:  [[7709 7710]] Out:  [[7711 7712]] Out:  [[7713 7714]] Out:  [[7715 7716]] Out:  [[7717 7718]] Out:  [[7719 7720]] Out:  [[7721 7722]] Out:  [[7723 7724]] Out:  [[7725 7726]] Out:  [[7727 7728]] Out

/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 7785 and 7787 respectively
Out:  [[7787 7788]] Out:  [[7787 7788]] Out:  [[7789 7790]] Out:  [[7791 7792]] Out:  [[7793 7794]] Out:  [[7795 7796]] Out:  [[7797 7798]] Out:  [[7799 7800]] Out:  [[7801 7802]] Out:  [[7803 7804]] Out:  [[7805 7806]] Out:  [[7807 7808]] Out:  [[7809 7810]] Out:  [[7811 7812]] Out:  [[7813 7814]] Out:  [[7815 7816]] Out:  [[7817 7818]] Out:  [[7819 7820]] Out:  [[7821 7822]] Out:  [[7823 7824]] Out:  [[7825 7826]] Out:  [[7827 7828]] Out:  [[7829 7830]] Out:  [[7831 7832]] Out:  [[7833 7834]] Out:  [[7835 7836]] Out:  [[7837 7838]] Out:  [[7839 7840]] Out:  [[7841 7842]] Out:  [[7843 7844]] Out:  [[7845 7846]] Out:  [[7847 7848]] Out:  [[7849 7850]] Out:  [[7851 7852]] Out:  [[7853 7854]] Out:  [[7855 7856]] Out:  [[7857 7858]] Out:  [[7859 7860]] Out:  [[7861 7862]] Out:  [[7863 7864]] Out:  [[7865 7866]] Out:  [[7867 7868]] Out:  [[7869 7870]] Out:  [[7871 7872]] Out:  [[7873 7874]] Out

...

Long rollout will begin with input and produce output from time index 9683 and 9685 respectively
Out:  [[9685 9686]] Out:  [[9685 9686]] Out:  [[9687 9688]] Out:  [[9689 9690]] Out:  [[9691 9692]] Out:  [[9693 9694]] Out:  [[9695 9696]] Out:  [[9697 9698]] Out:  [[9699 9700]] Out:  [[9701 9702]] Out:  [[9703 9704]] Out:  [[9705 9706]] Out:  [[9707 9708]] Out:  [[9709 9710]] Out:  [[9711 9712]] Out:  [[9713 9714]] Out:  [[9715 9716]] Out:  [[9717 9718]] Out:  [[9719 9720]] Out:  [[9721 9722]] Out:  [[9723 9724]] Out:  [[9725 9726]] Out:  [[9727 9728]] Out:  [[9729 9730]] Out:  [[9731 9732]] Out:  [[9733 9734]] Out:  [[9735 9736]] Out:  [[9737 9738]] Out:  [[9739 9740]] Out:  [[9741 9742]] Out:  [[9743 9744]] Out:  [[9745 9746]] Out:  [[9747 9748]] Out:  [[9749 9750]] Out:  [[9751 9752]] Out:  [[9753 9754]] Out:  [[9755 9756]] Out:  [[9757 9758]] Out:  [[9759 9760]] Out:  [[9761 9762]] Out:  [[9763 9764]] Out:  [[9765 9766]] Out:  [[9767 9768]] Out:  [[9769 9770]] Out:  [[9771 9772]] Out

/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 9829 and 9831 respectively
Out:  [[9831 9832]] Out:  [[9831 9832]] Out:  [[9833 9834]] Out:  [[9835 9836]] Out:  [[9837 9838]] Out:  [[9839 9840]] Out:  [[9841 9842]] Out:  [[9843 9844]] Out:  [[9845 9846]] Out:  [[9847 9848]] Out:  [[9849 9850]] Out:  [[9851 9852]] Out:  [[9853 9854]] Out:  [[9855 9856]] Out:  [[9857 9858]] Out:  [[9859 9860]] Out:  [[9861 9862]] Out:  [[9863 9864]] Out:  [[9865 9866]] Out:  [[9867 9868]] Out:  [[9869 9870]] Out:  [[9871 9872]] Out:  [[9873 9874]] Out:  [[9875 9876]] Out:  [[9877 9878]] Out:  [[9879 9880]] Out:  [[9881 9882]] Out:  [[9883 9884]] Out:  [[9885 9886]] Out:  [[9887 9888]] Out:  [[9889 9890]] Out:  [[9891 9892]] Out:  [[9893 9894]] Out:  [[9895 9896]] Out:  [[9897 9898]] Out:  [[9899 9900]] Out:  [[9901 9902]] Out:  [[9903 9904]] Out:  [[9905 9906]] Out:  [[9907 9908]] Out:  [[9909 9910]] Out:  [[9911 9912]] Out:  [[9913 9914]] Out:  [[9915 9916]] Out:  [[9917 9918]] Out

/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 9975 and 9977 respectively
Out:  [[9977 9978]] Out:  [[9977 9978]] Out:  [[9979 9980]] Out:  [[9981 9982]] Out:  [[9983 9984]] Out:  [[9985 9986]] Out:  [[9987 9988]] Out:  [[9989 9990]] Out:  [[9991 9992]] Out:  [[9993 9994]] Out:  [[9995 9996]] Out:  [[9997 9998]] Out:  [[ 9999 10000]] Out:  [[10001 10002]] Out:  [[10003 10004]] Out:  [[10005 10006]] Out:  [[10007 10008]] Out:  [[10009 10010]] Out:  [[10011 10012]] Out:  [[10013 10014]] Out:  [[10015 10016]] Out:  [[10017 10018]] Out:  [[10019 10020]] Out:  [[10021 10022]] Out:  [[10023 10024]] Out:  [[10025 10026]] Out:  [[10027 10028]] Out:  [[10029 10030]] Out:  [[10031 10032]] Out:  [[10033 10034]] Out:  [[10035 10036]] Out:  [[10037 10038]] Out:  [[10039 10040]] Out:  [[10041 10042]] Out:  [[10043 10044]] Out:  [[10045 10046]] Out:  [[10047 10048]] Out:  [[10049 10050]] Out:  [[10051 10052]] Out:  [[10053 10054]] Out:  [[10055 10056]] Out:  [[10057 10058]] Out